# Module 3 — Data Cleaning
All cleaning lives in the reusable `src/clean.py::clean()` so the exact same
logic runs here and in the deployment API (no train/serve skew). This notebook
documents every decision with **before/after** evidence.

In [1]:
import sys
from pathlib import Path
here = Path.cwd()
root = here if (here / "src").exists() else here.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
from config import ENRICHED_DATA_PATH, DATA_PROCESSED, FRAUD_TYPES
from clean import clean, add_quality_flags

raw = pd.read_parquet(ENRICHED_DATA_PATH)
print(f"BEFORE cleaning: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

BEFORE cleaning: 6,362,620 rows x 23 columns


## 1. Integrity baseline (before)
PaySim is structurally clean; we confirm and document it rather than assume it.

In [2]:
print("missing values total :", int(raw.isna().sum().sum()))
print("duplicate rows        :", int(raw.duplicated().sum()))
print("amount <= 0           :", int((raw['amount'] <= 0).sum()))
print("step out of [1,743]   :", int(((raw['step'] < 1) | (raw['step'] > 743)).sum()))

missing values total : 0
duplicate rows        : 0
amount <= 0           : 16
step out of [1,743]   : 0


## 2. The real quality issue: zero balances are *missing*, not zero
PaySim does not track merchant balances, and many destination balances are
untracked — they appear as `0` while `amount > 0`. Imputing or trusting these
zeros would corrupt the balance features. We instead expose them as explicit
flags and let the model use the "unknown" pattern.

In [3]:
flagged = add_quality_flags(raw)
print("dest_balance_unknown rows :", int(flagged['dest_balance_unknown'].sum()),
      f"({flagged['dest_balance_unknown'].mean()*100:.1f}%)")
print("orig_balance_unknown rows :", int(flagged['orig_balance_unknown'].sum()),
      f"({flagged['orig_balance_unknown'].mean()*100:.1f}%)")

# does the 'unknown' pattern itself carry fraud signal?
print("\nfraud rate by dest_balance_unknown:")
display(flagged.groupby('dest_balance_unknown')['isFraud'].mean().mul(100).round(4).rename('fraud_rate_pct'))

dest_balance_unknown rows : 2317276 (36.4%)
orig_balance_unknown rows : 2088969 (32.8%)

fraud rate by dest_balance_unknown:


dest_balance_unknown
0    0.1024
1    0.1756
Name: fraud_rate_pct, dtype: float64

## 3. Outliers: retained deliberately, not winsorized
`amount` is extremely right-skewed. We do **not** cap/winsorize it, for two
reasons that don't depend on which direction the amount–fraud relationship runs:
(1) the tree models we use are scale-robust, and (2) any cap is an arbitrary
threshold that risks discarding real signal. For the linear baseline we rely on
the `log_amount` transform instead. We remove only genuinely invalid rows
(`amount<=0`). Fraud rate across amount deciles on this data:

In [4]:
amount_decile = pd.qcut(raw['amount'], 10, labels=False, duplicates='drop')
print("fraud rate by amount decile (%):")
display(raw.groupby(amount_decile)['isFraud'].mean().mul(100).round(4).rename('fraud_rate_pct'))
print("=> the tails are not noise to be clipped; we retain them and let the "
      "model decide. (Direction of the amount effect is read off the real data.)")

fraud rate by amount decile (%):


amount
0    0.0233
1    0.0201
2    0.0233
3    0.0574
4    0.0970
5    0.0930
6    0.0913
7    0.0791
8    0.1130
9    0.6934
Name: fraud_rate_pct, dtype: float64

=> the tails are not noise to be clipped; we retain them and let the model decide. (Direction of the amount effect is read off the real data.)


## 4. Apply cleaning + document before/after
`clean()` removes duplicates / invalid rows, adds quality flags, scopes to the
fraud-bearing types (TRANSFER, CASH_OUT), and drops the leakage column
`isFlaggedFraud`. ID columns are **kept** — Module 4 needs them for velocity
features and drops them right before modelling.

In [5]:
clean_df, report = clean(raw)

rep = pd.Series(report, name="value").to_frame()
display(rep)

print(f"\nAFTER cleaning : {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns")
print(f"Rows removed   : {raw.shape[0] - clean_df.shape[0]:,}")

,value
rows_in,6362620
duplicates_removed,0
missing_core_removed,0
negative_amount_removed,0
amount_is_zero_flagged,16
invalid_step_removed,0
dest_balance_unknown,2317276
orig_balance_unknown,2088969
rows_dropped_by_type_scope,3592211
columns_dropped,[isFlaggedFraud]



AFTER cleaning : 2,770,409 rows x 25 columns
Rows removed   : 3,592,211


## 5. Class balance before vs after scoping

In [6]:
def rate(d): 
    return d['isFraud'].sum(), d['isFraud'].mean()*100
f0, r0 = rate(raw); f1, r1 = rate(clean_df)
print(f"before : {f0:,} fraud / {len(raw):,}  = {r0:.4f}%")
print(f"after  : {f1:,} fraud / {len(clean_df):,}  = {r1:.4f}%")
print(f"=> scoping to fraud-bearing types keeps ALL fraud and lifts the working "
      f"fraud rate {r0:.3f}% -> {r1:.3f}% (~{r1/r0:.1f}x), a better base for modelling.")

before : 8,213 fraud / 6,362,620  = 0.1291%
after  : 8,213 fraud / 2,770,409  = 0.2965%
=> scoping to fraud-bearing types keeps ALL fraud and lifts the working fraud rate 0.129% -> 0.296% (~2.3x), a better base for modelling.


## 6. Persist cleaned dataset

In [7]:
out = DATA_PROCESSED / "transactions_clean.parquet"
clean_df.to_parquet(out, index=False)
print("wrote", out)
print("columns:", list(clean_df.columns))

wrote /home/raven/Workspaces/ba/business-analysis-hust/feature_engineering/fraud-detection/data/processed/transactions_clean.parquet
columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'account_age_days', 'home_billing_country', 'home_device_id', 'device_id', 'is_new_device', 'browser_fingerprint', 'shipping_billing_mismatch', 'failed_payment_attempts', 'ip_country', 'ip_billing_distance_km', 'hour_of_day', 'is_night', 'amount_is_zero', 'dest_balance_unknown', 'orig_balance_unknown']


## 7. Cleaning decisions — summary (for the report)
| Decision | Action | Rationale |
|---|---|---|
| Duplicates / missing | remove (0 found) | Confirmed clean; guard kept for serving. |
| `amount <= 0`, invalid `step` | remove | Genuinely invalid records. |
| Extreme `amount` | **keep** | Fraud-informative; trees robust; winsorizing loses signal. |
| Zero balances | **flag, don't impute** | Zeros are untracked/missing; the "unknown" pattern is itself signal. |
| `isFlaggedFraud` | drop | Post-hoc rule, ~0 recall, not a real-time input (leakage risk). |
| Non-TRANSFER/CASH_OUT rows | drop (scope) | Fraud never occurs there; auto-approved at serving. Lifts fraud rate. |
| ID columns | keep for now | Needed for Module 4 velocity features; dropped before modelling. |

**Next (Module 4):** velocity/aggregate features per account, encode categoricals,
scale, split-then-resample (SMOTE/class-weights on the training fold only),
feature selection.